# Value Function approximation

- Introduction
- incremental Methods
- Batch methods



## Introduction

Reinforcement learning can be used to solve large problems, e.g.
- Backgammon: $10^20$ states
- Computer Go: $10^170$ states
- Helicopter: continuous state space

How can we scale up the model-free methods for prediction and control from the last two lectures?

## Value Function Approximation

- So far we have represented value function by a *lookup table*
  - Every state $s$ has an entry $V(s)$
  - Or every state-action pair $s, a$ has an entry $Q(s, a)$
- Problem with large MDPs:
  - There are too many states and/or actions to store in memory
  - It is too slow to learn the value of each state individually
- Solution for large MDPs:
  - Estimate value function with *function approximation*

$$\hat{v}(s, \mathbf{w}) \approx v_\pi(s)$$

$$\text{or } \hat{q}(s, a, \mathbf{w}) \approx q_\pi(s, a)$$

  - *Generalise* from seen states to unseen states
  - *Update* parameter $\mathbf{w}$ using MC or TD learning

## Types of Value Function Approximators

<img src="../image-101.png" width="400px"/><div>

Three common input/output configurations:

1. **State value**: $s \rightarrow \hat{v}(s, \mathbf{w})$
   - Input: state $s$, output: one scalar

2. **Action value (one action)**: $(s, a) \rightarrow \hat{q}(s, a, \mathbf{w})$
   - Input: state + action, output: one scalar

3. **Action value (all actions)**: $s \rightarrow \hat{q}(s, a_1, \mathbf{w}), \dots, \hat{q}(s, a_m, \mathbf{w})$
   - Input: state only, output: one value **per action**
   - Most efficient for action selection (single forward pass)
   - used in Atari games

In all cases, $\mathbf{w}$ are the learned parameters of the approximator (e.g. a neural network).

There are many function approximators, e.g.
- **Linear combinations of features**
- **Neural network**
- Decision tree
- Nearest neighbour
- Fourier/wavelet based
- ...

Here focus on differentiable function approximators

Require training method suitable for non-stationary, non-iid data



## Incremental methods



### Gradient descent

- Let $J(\mathbf{w})$ be a differentiable function of parameter vector $\mathbf{w}$
- Define the *gradient* of $J(\mathbf{w})$ to be

$$\nabla_\mathbf{w} J(\mathbf{w}) = \begin{pmatrix} \frac{\partial J(\mathbf{w})}{\partial w_1} \\ \vdots \\ \frac{\partial J(\mathbf{w})}{\partial w_n} \end{pmatrix}$$

- To find a local minimum of $J(\mathbf{w})$
- Adjust $\mathbf{w}$ in direction of -ve gradient

$$\Delta\mathbf{w} = -\frac{1}{2}\alpha \nabla_\mathbf{w} J(\mathbf{w})$$

where $\alpha$ is a step-size parameter

The $\frac{1}{2}$ often appears when $J(\mathbf{w})$ is a **mean squared error** loss:

$$J(\mathbf{w}) = \frac{1}{2}\sum_i (y_i - \hat{y}_i)^2$$

where the $\frac{1}{2}$ is introduced *in the loss itself* so that it cancels cleanly with the exponent when differentiating, giving tidy update rules. The slide likely just carried that factor into the update notation. It has no effect since $\alpha$ is a free hyperparameter anyway.

## Value Function Approx. By Stochastic Gradient Descent

- Goal: find parameter vector **w** minimising mean-squared error between approximate value fn $\hat{v}(s, \mathbf{w})$ and true value fn $v_\pi(s)$, (let's imagine we have this true $v_\pi$  like an oracle)

$$J(\mathbf{w}) = \mathbb{E}_\pi\left[(v_\pi(S) - \hat{v}(S, \mathbf{w}))^2\right]$$

- Gradient descent finds a local minimum

$$\Delta\mathbf{w} = -\frac{1}{2}\alpha \nabla_\mathbf{w} J(\mathbf{w})$$
$$= \alpha \mathbb{E}_\pi\left[(v_\pi(S) - \hat{v}(S, \mathbf{w})) \nabla_\mathbf{w} \hat{v}(S, \mathbf{w})\right]$$

- Stochastic gradient descent *samples* the gradient

$$\Delta\mathbf{w} = \alpha(v_\pi(S) - \hat{v}(S, \mathbf{w}))\nabla_\mathbf{w}\hat{v}(S, \mathbf{w})$$

- Expected update is equal to full gradient update

## Feature Vectors

- Represent state by a *feature vector*

$$\mathbf{x}(S) = \begin{pmatrix} \mathbf{x}_1(S) \\ \vdots \\ \mathbf{x}_n(S) \end{pmatrix}$$

- For example:
  - Distance of robot from landmarks
  - Trends in the stock market
  - Piece and pawn configurations in chess

## Linear Value Function Approximation

- Represent value function by a linear combination of features

$$\hat{v}(S, \mathbf{w}) = \mathbf{x}(S)^\top \mathbf{w} = \sum_{j=1}^{n} \mathbf{x}_j(S)\mathbf{w}_j$$

- Objective function is quadratic in parameters **w**

$$J(\mathbf{w}) = \mathbb{E}_\pi\left[(v_\pi(S) - \mathbf{x}(S)^\top\mathbf{w})^2\right]$$

- Stochastic gradient descent converges on *global* optimum (with linear combination)
- Update rule is particularly simple

$$\nabla_\mathbf{w}\hat{v}(S, \mathbf{w}) = \mathbf{x}(S)$$
$$\Delta\mathbf{w} = \alpha(v_\pi(S) - \hat{v}(S, \mathbf{w}))\mathbf{x}(S)$$

Update $=$ *step-size* $\times$ *prediction error* $\times$ *feature value*

## Table Lookup Features

- Table lookup is a special case of linear value function approximation
- Using *table lookup features*

$$\mathbf{x}^{table}(S) = \begin{pmatrix} \mathbf{1}(S = s_1) \\ \vdots \\ \mathbf{1}(S = s_n) \end{pmatrix}$$

Each entry is an indicator: $\mathbf{1}(S = s_i) = 1$ if the current state is $s_i$, and $0$ otherwise.
So $\mathbf{x}^{table}(S)$ is a one-hot vector — exactly one entry is $1$, all others are $0$.

- Parameter vector **w** gives value of each individual state

$$\hat{v}(S, \mathbf{w}) = \begin{pmatrix} \mathbf{1}(S = s_1) \\ \vdots \\ \mathbf{1}(S = s_n) \end{pmatrix} \cdot \begin{pmatrix} \mathbf{w}_1 \\ \vdots \\ \mathbf{w}_n \end{pmatrix}$$

Since the feature vector is one-hot, the dot product simply selects $\mathbf{w}_i$ for whichever state $S = s_i$ is active.
In other words, $\hat{v}(S, \mathbf{w}) = \mathbf{w}_i$ — each weight directly stores the value of one state.

This recovers **exact tabular RL**: no generalisation across states, every state has its own independent value estimate.
Function approximation reduces to a lookup table as a degenerate special case.

## Incremental Prediction Algorithms

- So far, we have assumed true value function $v_\pi(s)$ given by supervisor
- But in RL there is no supervisor, only rewards
- In practice, we substitute a *target* for $v_\pi(s)$

The key insight: we never know $v_\pi(s)$ exactly, so we replace it with a
bootstrapped or sampled estimate depending on the algorithm.
The update rule $\Delta\mathbf{w} = \alpha(\text{target} - \hat{v}(S_t, \mathbf{w}))\nabla_\mathbf{w}\hat{v}(S_t, \mathbf{w})$
is the same in all cases — only the target differs.

---

### Monte Carlo (MC)

Target is the actual return $G_t$:

$$\Delta\mathbf{w} = \alpha(G_t - \hat{v}(S_t, \mathbf{w}))\nabla_\mathbf{w}\hat{v}(S_t, \mathbf{w})$$

$G_t$ is the true sampled return from that episode. Unbiased but high variance —
you must wait until the episode ends to update.

---

### TD(0)

Target is the one-step bootstrapped estimate $R_{t+1} + \gamma\hat{v}(S_{t+1}, \mathbf{w})$:

$$\Delta\mathbf{w} = \alpha(R_{t+1} + \gamma\hat{v}(S_{t+1}, \mathbf{w}) - \hat{v}(S_t, \mathbf{w}))\nabla_\mathbf{w}\hat{v}(S_t, \mathbf{w})$$

Updates online after each step. Lower variance than MC but biased because the
target itself depends on the current (imperfect) **w**.

---

### TD($\lambda$)

Target is the $\lambda$-return $G_t^\lambda$, a geometric mixture of $n$-step returns:

$$\Delta\mathbf{w} = \alpha(G_t^\lambda - \hat{v}(S_t, \mathbf{w}))\nabla_\mathbf{w}\hat{v}(S_t, \mathbf{w})$$

$\lambda \in [0,1]$ interpolates between TD(0) ($\lambda=0$) and MC ($\lambda=1$),
trading off bias and variance. In practice implemented efficiently via eligibility traces.

### Monte-Carlo with Value Function Approximation

- Return $G_t$ is an unbiased, noisy sample of true value $v_\pi(S_t)$
- Can therefore apply supervised learning to "training data":

$$\langle S_1, G_1\rangle, \langle S_2, G_2\rangle, ..., \langle S_T, G_T\rangle$$

Each episode generates a sequence of (state, return) pairs. Since $G_t$ is an
unbiased estimate of $v_\pi(S_t)$, this is exactly like regression with noisy labels —
we can treat it as a standard supervised learning problem.

---

#### Linear Monte-Carlo Policy Evaluation

$$\Delta\mathbf{w} = \alpha(G_t - \hat{v}(S_t, \mathbf{w}))\nabla_\mathbf{w}\hat{v}(S_t, \mathbf{w})$$
$$= \alpha(G_t - \hat{v}(S_t, \mathbf{w}))\mathbf{x}(S_t)$$

The second line uses the fact that for linear approximation
$\nabla_\mathbf{w}\hat{v}(S_t, \mathbf{w}) = \mathbf{x}(S_t)$ 

The **gradient is just the feature vector**: 

$$\hat{v}(S, \mathbf{w}) = \mathbf{x}(S)^\top \mathbf{w} = \sum_{j=1}^n x_j(S) w_j$$

Taking the gradient with respect to $\mathbf{w}$:

$$\frac{\partial \hat{v}}{\partial w_j} = x_j(S)$$

So:

$$\nabla_\mathbf{w}\hat{v}(S, \mathbf{w}) = \mathbf{x}(S)$$

The features $\mathbf{x}(S)$ don't depend on $\mathbf{w}$ — they're fixed functions of the state.
The weights $\mathbf{w}$ appear linearly, so differentiating just peels them off,
leaving the feature vector. Same reason $\frac{d}{dw}(aw) = a$.



The update has a clean interpretation:
- $(G_t - \hat{v}(S_t, \mathbf{w}))$ is the **prediction error** — how wrong our estimate was
- $\mathbf{x}(S_t)$ is the **feature vector** — which dimensions of **w** to adjust
- $\alpha$ controls the **step size**

Each weight $w_j$ is nudged proportionally to both the error and the feature $x_j(S_t)$.

---

- Monte-Carlo evaluation converges to a **local optimum**
- Even when using non-linear value function approximation

> Note: convergence to a *local* (not global) optimum is guaranteed for non-linear
> approximators. For linear approximation, the objective is quadratic so SGD finds
> the global optimum.

### TD Learning with Value Function Approximation

- The TD-target $R_{t+1} + \gamma\hat{v}(S_{t+1}, \mathbf{w})$ is a *biased* sample of true value $v_\pi(S_t)$
- Can still apply supervised learning to "training data":

$$\langle S_1, R_2 + \gamma\hat{v}(S_2, \mathbf{w})\rangle, \langle S_2, R_3 + \gamma\hat{v}(S_3, \mathbf{w})\rangle, ..., \langle S_{T-1}, R_T\rangle$$

Unlike MC, the target is not the true return but a **bootstrap estimate** — it uses the
current approximation $\hat{v}$ to estimate future value. This introduces bias because
the target itself depends on the parameters **w** being learned. Despite this, TD still works.

---

#### Linear TD(0)

$$\Delta\mathbf{w} = \alpha(R + \gamma\hat{v}(S', \mathbf{w}) - \hat{v}(S, \mathbf{w}))\nabla_\mathbf{w}\hat{v}(S, \mathbf{w})$$
$$= \alpha\delta\mathbf{x}(S)$$

Where $\delta = R + \gamma\hat{v}(S', \mathbf{w}) - \hat{v}(S, \mathbf{w})$ is the **TD error** — the difference
between the bootstrapped target and the current estimate.

Again, $\nabla_\mathbf{w}\hat{v}(S, \mathbf{w}) = \mathbf{x}(S)$ for linear approximation, so the update
simplifies to: scale the feature vector by the TD error and step size.

> Note: we do **not** differentiate through the TD target $R + \gamma\hat{v}(S', \mathbf{w})$ —
> it is treated as a fixed constant when computing the gradient. This is what makes
> TD a *semi-gradient* method.

---

- Linear TD(0) converges **(close)** to global optimum

> "Close" because TD does not minimise the MSE directly — it converges to the
> **TD fixed point**, which is near but not identical to the true least-squares solution.
> The error at the TD fixed point is bounded by a factor of $\frac{1}{1-\gamma}$ times
> the minimum possible MSE.

### TD(λ) with Value Function Approximation

- The $\lambda$-return $G_t^\lambda$ is also a biased sample of true value $v_\pi(s)$
- Can again apply supervised learning to "training data":

$$\left\langle S_1, G_1^\lambda\right\rangle, \left\langle S_2, G_2^\lambda\right\rangle, ..., \left\langle S_{T-1}, G_{T-1}^\lambda\right\rangle$$

---

#### Forward View Linear TD(λ)

$$\Delta\mathbf{w} = \alpha(G_t^\lambda - \hat{v}(S_t, \mathbf{w}))\nabla_\mathbf{w}\hat{v}(S_t, \mathbf{w})$$
$$= \alpha(G_t^\lambda - \hat{v}(S_t, \mathbf{w}))\mathbf{x}(S_t)$$

The $\lambda$-return $G_t^\lambda$ is a weighted average of all $n$-step returns:
- $\lambda=0$: reduces to TD(0), full bootstrap
- $\lambda=1$: reduces to MC, no bootstrap

The forward view looks **ahead** to future rewards to compute $G_t^\lambda$,
so it requires the full episode before updating — not practical online.

---

#### Backward View Linear TD(λ)

$$\delta_t = R_{t+1} + \gamma\hat{v}(S_{t+1}, \mathbf{w}) - \hat{v}(S_t, \mathbf{w})$$
$$E_t = \gamma\lambda E_{t-1} + \mathbf{x}(S_t)$$
$$\Delta\mathbf{w} = \alpha\delta_t E_t$$

The backward view is the **online, implementable equivalent** of the forward view.

- $\delta_t$ is the one-step **TD error** at time $t$
- $E_t$ is the **eligibility trace** — a memory vec

### Control with value function approximation

<img src="../image-102.png" width="400px">

- Policy evaluation: Approximate policy evaluation, $\hat{q}(\cdot, \cdot, \mathbf{w}) \approx q_\pi$
- Policy improvement: $\epsilon$-greedy policy improvement

### Action-Value Function Approximation

- Approximate the action-value function

$$\hat{q}(S, A, \mathbf{w}) \approx q_\pi(S, A)$$

Same idea as value function approximation, but now we approximate $q(S, A)$
over both state **and** action — needed for control (policy improvement requires
comparing action values).

---

#### Objective

Minimise mean-squared error between approximate action-value fn $\hat{q}(S, A, \mathbf{w})$
and true action-value fn $q_\pi(S, A)$:

$$J(\mathbf{w}) = \mathbb{E}_\pi\left[(q_\pi(S, A) - \hat{q}(S, A, \mathbf{w}))^2\right]$$

---

#### Stochastic Gradient Descent

$$-\frac{1}{2}\nabla_\mathbf{w}J(\mathbf{w}) = (q_\pi(S, A) - \hat{q}(S, A, \mathbf{w}))\nabla_\mathbf{w}\hat{q}(S, A, \mathbf{w})$$

$$\Delta\mathbf{w} = \alpha(q_\pi(S, A) - \hat{q}(S, A, \mathbf{w}))\nabla_\mathbf{w}\hat{q}(S, A, \mathbf{w})$$

Identical in structure to value function approximation. The update:
- moves **w** to reduce the error between target $q_\pi(S,A)$ and prediction $\hat{q}(S,A,\mathbf{w})$
- scales the step by the gradient of $\hat{q}$ w.r.t. **w**

In practice, $q_\pi(S, A)$ is unknown and replaced by a target:
- **MC**: $G_t$
- **TD(0)**: $R_{t+1} + \gamma \hat{q}(S_{t+1}, A_{t+1}, \mathbf{w})$
- **TD(λ)**: $q_t^\lambda$

### Linear Action-Value Function Approximation

- Represent state *and* action by a *feature vector*

$$\mathbf{x}(S, A) = \begin{pmatrix} \mathbf{x}_1(S, A) \\ \vdots \\ \mathbf{x}_n(S, A) \end{pmatrix}$$

Each feature $\mathbf{x}_j(S, A)$ captures some aspect of the (state, action) pair —
e.g. distance to goal combined with action direction, or tile coding over state-action space.

---

#### Linear Action-Value Function

$$\hat{q}(S, A, \mathbf{w}) = \mathbf{x}(S, A)^\top \mathbf{w} = \sum_{j=1}^{n} \mathbf{x}_j(S, A)\mathbf{w}_j$$

The action-value is a weighted sum of features. Each weight $w_j$ captures
how much feature $j$ contributes to the value of taking action $A$ in state $S$.

---

#### Stochastic Gradient Descent Update

$$\nabla_\mathbf{w}\hat{q}(S, A, \mathbf{w}) = \mathbf{x}(S, A)$$

$$\Delta\mathbf{w} = \alpha(q_\pi(S, A) - \hat{q}(S, A, \mathbf{w}))\mathbf{x}(S, A)$$

As with linear value approximation, the gradient simplifies to the feature vector
since **w** appears linearly. The update rule is:

- $(q_\pi(S,A) - \hat{q}(S,A,\mathbf{w}))$: **prediction error** for this (state, action) pair
- $\mathbf{x}(S, A)$: **feature vector** indicating which weights to adjust
- $\alpha$: **step size**

In practice $q_\pi(S,A)$ is substituted with a TD or MC target, exactly as in the
value function case.

## Incremental Control Algorithms

Like prediction, we must substitute a *target* for $q_\pi(S, A)$.
The structure is identical to the prediction case — only the target changes.

---

#### Monte Carlo

Target is the actual return $G_t$:

$$\Delta\mathbf{w} = \alpha(G_t - \hat{q}(S_t, A_t, \mathbf{w}))\nabla_\mathbf{w}\hat{q}(S_t, A_t, \mathbf{w})$$

Unbiased but high variance. Must wait until episode end to update.

---

#### TD(0)

Target is the one-step bootstrap $R_{t+1} + \gamma\hat{q}(S_{t+1}, A_{t+1}, \mathbf{w})$:

$$\Delta\mathbf{w} = \alpha(R_{t+1} + \gamma\hat{q}(S_{t+1}, A_{t+1}, \mathbf{w}) - \hat{q}(S_t, A_t, \mathbf{w}))\nabla_\mathbf{w}\hat{q}(S_t, A_t, \mathbf{w})$$

This is **Sarsa with function approximation** — the next action $A_{t+1}$ is drawn
from the current policy (on-policy). Lower variance than MC but biased since the
target depends on **w**.

---

#### Forward-View TD(λ)

Target is the action-value $\lambda$-return $q_t^\lambda$:

$$\Delta\mathbf{w} = \alpha(q_t^\lambda - \hat{q}(S_t, A_t, \mathbf{w}))\nabla_\mathbf{w}\hat{q}(S_t, A_t, \mathbf{w})$$

$q_t^\lambda$ blends $n$-step returns with geometric weights $(1-\lambda)\lambda^{n-1}$.
Requires full episode — not practical online.

---

#### Backward-View TD(λ)

$$\delta_t = R_{t+1} + \gamma\hat{q}(S_{t+1}, A_{t+1}, \mathbf{w}) - \hat{q}(S_t, A_t, \mathbf{w})$$
$$E_t = \gamma\lambda E_{t-1} + \nabla_\mathbf{w}\hat{q}(S_t, A_t, \mathbf{w})$$
$$\Delta\mathbf{w} = \alpha\delta_t E_t$$

The online equivalent of forward-view TD(λ):
- $\delta_t$: **TD error** — how wrong the current prediction was
- $E_t$: **eligibility trace** — accumulates gradients of visited (state, action) pairs,
  decaying by $\gamma\lambda$ each step. Assigns credit to recently visited pairs.
- Update all weights at every step, weighted by their trace

> For linear approximation, $\nabla_\mathbf{w}\hat{q}(S_t, A_t, \mathbf{w}) = \mathbf{x}(S_t, A_t)$,
> so the trace simply accumulates feature vectors: $E_t = \gamma\lambda E_{t-1} + \mathbf{x}(S_t, A_t)$.